# CALIBRE: Implementation Notebook

Companion notebook for *Beyond AUC: Calibrated and Risk-Controlled Link Prediction with Structural and Variational Graph Embeddings* (Shivnaran S, Laxman V, Saripiralla Dushyanth Reddy, PSG College of Technology, Coimbatore).

The notebook runs the whole study in order:

1. Reproduces the ten-node toy example (Adamic–Adar scores 0.00 for a real hidden edge and 1.44 for a hard non-edge).
2. Builds the nested observation/supervision split and three negative samplers: uniform, distance-2 (hard), and distance-2 with a joint-degree match.
3. Implements the classical indices, node2vec (skip-gram with biased walks), and GAE/VGAE from scratch in PyTorch.
4. Derives and numerically verifies Proposition 1 (exact moments of the inner-product score) and Proposition 2 (probit-moderated probability).
5. Assembles CALIBRE from four evidence families with gradient boosting.
6. Calibrates with monotone maps and reports ECE, Brier, NLL, and the Cox slope and intercept.
7. Controls the false discovery rate with conformal p-values and Benjamini–Hochberg, including a degree-stratified (Mondrian) variant.
8. Runs the experiment grid, the paired statistics (Wilcoxon, Holm, bootstrap, rank-biserial), and produces tables and figures.

**How to run.** Install `numpy pandas networkx scipy scikit-learn torch matplotlib` and, for the Planetoid citation graphs, `torch-geometric`. The SNAP graphs are downloaded automatically. `FAST = True` runs a smoke test in a few minutes on a laptop CPU using a synthetic graph plus Cora and CiteSeer. Set `FAST = False` for the full study. Numbers for the paper must be regenerated from this notebook; see the last section for the places where this implementation deliberately differs from the paper.

In [ ]:
# ---------------------------------------------------------------- 0. Setup
import math, time, warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import networkx as nx
import scipy.sparse as sp
from scipy import stats
from scipy.optimize import minimize
from scipy.special import expit, logit
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
from sklearn.ensemble import HistGradientBoostingClassifier

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FAST = True   # True: smoke test (minutes). False: the full study used for the paper.

CONFIG = dict(
    datasets=(["synthetic", "cora", "citeseer"] if FAST else
              ["cora", "citeseer", "pubmed", "facebook", "ca-GrQc", "email-EU", "bitcoin-otc"]),
    regimes=["random"] if FAST else ["random", "cold_start", "temporal"],
    neg_modes=["uniform", "hard", "hard_degree"],
    seeds=[0, 1] if FAST else [0, 1, 2, 3, 4],
    test_frac=0.10, cal_frac=0.05, obs_frac_of_train=0.80,
    n2v=dict(dim=128, num_walks=2 if FAST else 10, walk_len=40 if FAST else 80,
             window=10, neg=5, max_pairs=300_000 if FAST else 3_000_000,
             epochs=1, lr=0.01, batch=4096, p=1.0, q=1.0),
    gae=dict(hidden=32, dim=16, epochs=100 if FAST else 200, lr=0.01),
    alphas=[0.05, 0.10, 0.20, 0.30],
    hits_k=50,
    out_dir=Path("results"),
)
CONFIG["out_dir"].mkdir(exist_ok=True)
print("device:", DEVICE, "| FAST:", FAST, "| datasets:", CONFIG["datasets"])

## 1. Datasets

Each graph is returned as an undirected `networkx.Graph` with nodes relabelled `0..n-1`, an optional attribute matrix `X` (`None` for featureless graphs, which use the identity design matrix as in the VGAE paper), and an optional dictionary of first-contact times for the temporal regime. A `synthetic` stochastic block model is included so the whole pipeline runs offline.

In [ ]:
SNAP_URLS = {
    "facebook": "https://snap.stanford.edu/data/facebook_combined.txt.gz",
    "ca-GrQc": "https://snap.stanford.edu/data/ca-GrQc.txt.gz",
    "email-EU": "https://snap.stanford.edu/data/email-Eu-core-temporal.txt.gz",
    "bitcoin-otc": "https://snap.stanford.edu/data/soc-sign-bitcoinotc.csv.gz",
}

def _graph_from_edges(u, v, t=None):
    """Relabel nodes to 0..n-1, drop self-loops, keep the first-contact time of each edge."""
    ids = {x: i for i, x in enumerate(pd.unique(np.concatenate([u, v])))}
    U = np.array([ids[x] for x in u]); V = np.array([ids[x] for x in v])
    times = t if t is not None else np.zeros(len(U))
    ts = {}
    for a, b, tt in zip(U, V, times):
        if a == b:
            continue
        key = (int(min(a, b)), int(max(a, b)))
        ts[key] = min(ts.get(key, np.inf), float(tt))
    G = nx.Graph(); G.add_nodes_from(range(len(ids))); G.add_edges_from(ts.keys())
    return G, (ts if t is not None else None)

def synthetic_sbm(n=600, k=4, seed=0):
    """Offline stand-in: 4-block stochastic block model with noisy block-indicator attributes."""
    rng = np.random.default_rng(seed)
    block = rng.integers(k, size=n)
    P = np.full((k, k), 0.01); np.fill_diagonal(P, 0.08)
    G = nx.Graph(); G.add_nodes_from(range(n))
    for i in range(n):
        for j in range(i + 1, n):
            if rng.random() < P[block[i], block[j]]:
                G.add_edge(i, j)
    X = np.eye(k)[block] + 0.5 * rng.standard_normal((n, k))
    ts = {(min(e), max(e)): float(rng.random()) for e in G.edges()}
    return G, X, ts

def load_graph(name, root="data"):
    """Return (G, X, ts). X is None for featureless graphs; ts is None unless timestamps exist."""
    if name == "synthetic":
        return synthetic_sbm()
    if name in ("cora", "citeseer", "pubmed"):
        from torch_geometric.datasets import Planetoid   # pip install torch-geometric
        d = Planetoid(root=f"{root}/Planetoid", name=name.capitalize())[0]
        ei = d.edge_index.numpy()
        G, _ = _graph_from_edges(ei[0], ei[1])
        return G, d.x.numpy(), None
    url = SNAP_URLS[name]
    if name == "bitcoin-otc":
        df = pd.read_csv(url, header=None, names=["u", "v", "r", "t"])
        G, ts = _graph_from_edges(df.u.values, df.v.values, df.t.values)
        return G, None, ts
    cols = ["u", "v"] + (["t"] if name == "email-EU" else [])
    df = pd.read_csv(url, sep=r"\s+", header=None, comment="#", names=cols)
    if name == "email-EU":
        G, ts = _graph_from_edges(df.u.values, df.v.values, df.t.values)
        return G, None, ts
    G, _ = _graph_from_edges(df.u.values, df.v.values)
    return G, None, None

G_demo, X_demo, ts_demo = load_graph("synthetic")
print("synthetic:", G_demo.number_of_nodes(), "nodes,", G_demo.number_of_edges(), "edges, attributes:", X_demo.shape[1])

## 2. The ten-node example

Two tight clusters joined through node E. The real hidden edge D–J is removed from the observed graph, and C–F is a true non-edge two steps apart through E. Adamic–Adar gives the real edge 0 and the fake edge 1.44, so the textbook heuristic ranks the non-edge above the missing edge.

In [ ]:
TOY = nx.Graph([("A","B"),("A","C"),("B","C"),("C","D"),("A","J"),("C","E"),
                ("E","F"),("F","G"),("F","H"),("G","H"),("H","I"),("G","I")])
HIDDEN_REAL, HARD_FAKE = ("D", "J"), ("C", "F")

def aa_manual(G, u, v):
    return sum(1.0 / math.log(G.degree(w)) for w in nx.common_neighbors(G, u, v))

toy_rows = []
for pair, kind in [(HIDDEN_REAL, "real hidden edge"), (HARD_FAKE, "hard non-edge")]:
    toy_rows.append(dict(pair="-".join(pair), kind=kind,
                         common=sorted(nx.common_neighbors(TOY, *pair)),
                         adamic_adar=round(aa_manual(TOY, *pair), 4),
                         networkx_check=round(list(nx.adamic_adar_index(TOY, [pair]))[0][2], 4)))
toy_df = pd.DataFrame(toy_rows)
print(toy_df.to_string(index=False))
assert abs(aa_manual(TOY, *HARD_FAKE) - 1 / math.log(2)) < 1e-12 and aa_manual(TOY, *HIDDEN_REAL) == 0

pos_xy = {"A":(-2,0.8),"B":(-2,-0.8),"C":(-1,0),"D":(-0.3,-0.9),"J":(-1.6,1.9),
          "E":(0.9,0),"F":(1.8,0),"G":(2.7,0.8),"H":(2.7,-0.8),"I":(3.6,0)}
fig, ax = plt.subplots(figsize=(7, 4))
nx.draw_networkx_edges(TOY, pos_xy, ax=ax, edge_color="grey", width=2)
nx.draw_networkx_edges(nx.Graph([HIDDEN_REAL]), pos_xy, ax=ax, edge_color="green", style="dashed", width=2)
nx.draw_networkx_edges(nx.Graph([HARD_FAKE]), pos_xy, ax=ax, edge_color="red", style="dashed", width=2)
nx.draw_networkx_nodes(TOY, pos_xy, ax=ax, node_color="lightblue", node_size=500)
nx.draw_networkx_labels(TOY, pos_xy, ax=ax)
ax.set_title("Ten-node example: green = hidden real edge, red = hard non-edge")
ax.axis("off"); plt.show()

## 3. Nested observation/supervision split and negative sampling

**Split.** Edges are partitioned into a test block (10%), a calibration block (5%), and a training block (85%). The training block is split again into an observation set (80%) that builds the graph the methods may see, and a supervision set (20%) that supplies classifier training examples. Those supervision edges are absent from the observed graph, so their features look as hard as test features. Three regimes are implemented: `random`, `cold_start` (test edges drawn with probability proportional to 1/min(deg)², by weighted reservoir ordering), and `temporal` (earliest edges for training, latest for test).

**Negatives.** `uniform`: any non-adjacent pair. `hard`: pairs at shortest-path distance exactly two in the observed graph, drawn by a two-step random walk with rejection. `hard_degree`: distance-2 pairs resampled so that each negative falls into the same joint-degree cell (⌊log₂⌋ of the smaller and larger endpoint degree) as a positive. That is the control for degree confounding.

In [ ]:
def split_edges(G, regime, rng, cfg, ts=None):
    """Nested observation/supervision split. Returns canonical (k,2) edge arrays."""
    E = np.sort(np.array(list(G.edges()), dtype=np.int64), axis=1)
    m = len(E)
    if regime == "random":
        order = rng.permutation(m)
    elif regime == "temporal":
        tt = np.array([ts[(int(a), int(b))] for a, b in E])
        order = np.argsort(tt, kind="stable")                       # earliest first
    elif regime == "cold_start":
        deg = np.array([G.degree(i) for i in range(G.number_of_nodes())], dtype=float)
        w = 1.0 / np.minimum(deg[E[:, 0]], deg[E[:, 1]]) ** 2
        keys = rng.random(m) ** (1.0 / w)                            # Efraimidis–Spirakis ordering
        order = np.argsort(-keys)
    else:
        raise ValueError(regime)

    n_test, n_cal = int(cfg["test_frac"] * m), int(cfg["cal_frac"] * m)
    if regime == "temporal":
        test = E[order[m - n_test:]]
        cal = E[order[m - n_test - n_cal:m - n_test]]
        train = E[order[:m - n_test - n_cal]]
    else:
        test = E[order[:n_test]]
        cal = E[order[n_test:n_test + n_cal]]
        train = E[order[n_test + n_cal:]]

    k = int(cfg["obs_frac_of_train"] * len(train))
    if regime == "temporal":
        obs, sup = train[:k], train[k:]                              # observation precedes supervision
    else:
        perm = rng.permutation(len(train))
        obs, sup = train[perm[:k]], train[perm[k:]]
    return dict(test=test, cal=cal, sup=sup, obs=obs)

def build_graph(n, edges):
    H = nx.Graph(); H.add_nodes_from(range(n))
    H.add_edges_from(map(tuple, np.asarray(edges).tolist()))
    return H

def _key(u, v):
    return (u, v) if u < v else (v, u)

def sample_negatives(count, mode, G, Gobs, rng, used, pos):
    """Draw `count` non-edges of the full graph G that are not in `used`.
    mode: 'uniform' | 'hard' (distance exactly 2 in Gobs) | 'hard_degree' (distance 2, degree-matched to pos)."""
    n = G.number_of_nodes()
    if mode == "uniform":
        out = []
        while len(out) < count:
            u, v = (int(x) for x in rng.integers(n, size=2))
            k = _key(u, v)
            if u != v and not G.has_edge(u, v) and k not in used:
                used.add(k); out.append(k)
        return np.array(out, dtype=np.int64).reshape(-1, 2)

    nbrs = [np.array(sorted(Gobs.neighbors(x)), dtype=np.int64) for x in range(n)]
    anchors = np.array([x for x in range(n) if len(nbrs[x]) > 0])

    def draw_hard(num):
        out, seen, tries = [], set(), 0
        while len(out) < num and tries < 100 * num + 10_000:
            tries += 1
            u = int(rng.choice(anchors)); w = int(rng.choice(nbrs[u])); v = int(rng.choice(nbrs[w]))
            k = _key(u, v)
            if u == v or Gobs.has_edge(u, v) or G.has_edge(u, v) or k in used or k in seen:
                continue
            seen.add(k); out.append(k)
        return out

    if mode == "hard":
        out = draw_hard(count)
    elif mode == "hard_degree":
        def deg(x):
            return max(Gobs.degree(int(x)), 1)
        def cell(a, b):
            lo, hi = sorted((deg(a), deg(b)))
            return (int(math.log2(lo)), int(math.log2(hi)))
        need = Counter(cell(a, b) for a, b in pos)
        pool = draw_hard(20 * count)
        buckets = defaultdict(list)
        for a, b in pool:
            buckets[cell(a, b)].append((a, b))
        out = []
        for c, k in need.items():
            out += buckets[c][:k]
        chosen = set(out)
        short = count - len(out)
        if short > 0:
            print(f"  [hard_degree] {short} negatives could not be cell-matched; topping up")
            out += [p_ for p_ in pool if p_ not in chosen][:short]
    else:
        raise ValueError(mode)

    out = out[:count]
    if len(out) < count:
        print(f"  [{mode}] only {len(out)}/{count} negatives found")
    for k in out:
        used.add(k)
    return np.array(out, dtype=np.int64).reshape(-1, 2)

## 4. Classical indices on the observed graph

Shared-neighbour indices are computed from the sparse adjacency matrix: for weights `w`, the score of a pair is `Σ_w A[u,w] w_w A[w,v]`. The file implements 22 topological features, including Adamic–Adar, resource allocation, Jaccard, preferential attachment, the local path index (three-path count), and community and core statistics. The paper uses 27; the missing ones are straightforward additions.

In [ ]:
def pair_weighted(A, U, V, w):
    """sum_w A[u,w] * w[w] * A[w,v] for every pair (U[k], V[k])."""
    M = A @ sp.diags(w)
    return np.asarray(M[U].multiply(A[V]).sum(axis=1)).ravel()

def build_context(Gobs, seed):
    n = Gobs.number_of_nodes()
    A = sp.csr_matrix(nx.to_scipy_sparse_array(Gobs, nodelist=list(range(n)), format="csr", dtype=float))
    A2 = (A @ A).tocsr()
    deg = np.asarray(A.sum(1)).ravel()
    core_d = nx.core_number(Gobs)
    clust_d = nx.clustering(Gobs)
    comms = nx.community.louvain_communities(Gobs, seed=seed)
    comm = np.zeros(n, dtype=int); csize = np.zeros(n)
    for cid, c in enumerate(comms):
        for x in c:
            comm[x] = cid; csize[x] = len(c)
    return dict(A=A, A2=A2, deg=deg,
                core=np.array([core_d[i] for i in range(n)], dtype=float),
                clust=np.array([clust_d[i] for i in range(n)], dtype=float),
                comm=comm, csize=csize)

TOPO_NAMES = ["CN","Jaccard","AA","RA","PA","Salton","Sorensen","HPI","HDI","LHN","Path3",
              "deg_u","deg_v","deg_min","deg_max","core_u","core_v","same_comm",
              "comm_size_u","comm_size_v","clust_u","clust_v"]

def topo_features(pairs, ctx):
    A, A2, deg = ctx["A"], ctx["A2"], ctx["deg"]
    U, V = pairs[:, 0], pairs[:, 1]
    n = len(deg); eps = 1e-12
    cn = pair_weighted(A, U, V, np.ones(n))
    aa = pair_weighted(A, U, V, np.where(deg >= 2, 1.0 / np.log(np.maximum(deg, 2)), 0.0))
    ra = pair_weighted(A, U, V, np.where(deg >= 1, 1.0 / np.maximum(deg, 1), 0.0))
    du, dv = deg[U], deg[V]
    p3 = np.asarray(A2[U].multiply(A[V]).sum(axis=1)).ravel()   # (A^3)[u,v] for symmetric A
    cols = [cn, cn / (du + dv - cn + eps), aa, ra, du * dv,
            cn / np.sqrt(du * dv + eps), 2 * cn / (du + dv + eps),
            cn / np.maximum(np.minimum(du, dv), 1), cn / np.maximum(np.maximum(du, dv), 1),
            cn / (du * dv + eps), p3,
            du, dv, np.minimum(du, dv), np.maximum(du, dv),
            ctx["core"][U], ctx["core"][V],
            (ctx["comm"][U] == ctx["comm"][V]).astype(float),
            ctx["csize"][U], ctx["csize"][V], ctx["clust"][U], ctx["clust"][V]]
    return np.column_stack(cols).astype(float)

def stratum(pairs, deg):
    """Mondrian stratum: floor(log2) of the smaller endpoint degree."""
    d = np.maximum(deg[pairs], 1).min(axis=1)
    return np.floor(np.log2(d)).astype(int)

## 5. node2vec

Second-order biased random walks (return parameter `p`, in-out parameter `q`) generate node sequences. Skip-gram with negative sampling, trained in PyTorch, learns 128-dimensional embeddings. Pairs are subsampled to `max_pairs` to keep the cost bounded. Following the paper, `(p, q)` are tuned on the calibration split when `FAST = False`. Alias sampling is omitted for clarity; the walks are recomputed directly.

In [ ]:
PQ_GRID = [(1, 1), (1, 0.5), (1, 2), (0.25, 1), (4, 1)]

def node2vec_walks(Gobs, p, q, num_walks, walk_len, rng):
    n = Gobs.number_of_nodes()
    nbrs = [list(Gobs.neighbors(x)) for x in range(n)]
    nbr_set = [set(a) for a in nbrs]
    walks = []
    for _ in range(num_walks):
        for s in rng.permutation(n):
            walk = [int(s)]
            if not nbrs[s]:
                walks.append(walk); continue
            walk.append(int(rng.choice(nbrs[s])))
            while len(walk) < walk_len:
                cur, prev = walk[-1], walk[-2]
                cand = nbrs[cur]
                if not cand:
                    break
                wts = np.array([1 / p if x == prev else (1.0 if x in nbr_set[prev] else 1 / q)
                                for x in cand])
                idx = int(np.searchsorted(np.cumsum(wts), rng.random() * wts.sum(), side="right"))
                walk.append(cand[min(idx, len(cand) - 1)])
            walks.append(walk)
    return walks

def train_skipgram(walks, n, c, rng):
    L = max(len(w) for w in walks)
    W = np.full((len(walks), L), -1, dtype=np.int64)
    for i, w in enumerate(walks):
        W[i, :len(w)] = w
    pairs = []
    for d in range(1, c["window"] + 1):
        a, b = W[:, :-d], W[:, d:]
        ok = (a >= 0) & (b >= 0)
        pairs += [np.stack([a[ok], b[ok]], 1), np.stack([b[ok], a[ok]], 1)]
    P = np.concatenate(pairs)
    if len(P) > c["max_pairs"]:
        P = P[rng.choice(len(P), c["max_pairs"], replace=False)]
    tokens = W[W >= 0]
    noise = np.bincount(tokens, minlength=n).astype(float) ** 0.75
    noise = torch.tensor(noise / noise.sum(), dtype=torch.float32, device=DEVICE)

    emb_in = nn.Embedding(n, c["dim"]).to(DEVICE)
    emb_out = nn.Embedding(n, c["dim"]).to(DEVICE)
    nn.init.uniform_(emb_in.weight, -0.5 / c["dim"], 0.5 / c["dim"]); nn.init.zeros_(emb_out.weight)
    opt = torch.optim.Adam(list(emb_in.parameters()) + list(emb_out.parameters()), lr=c["lr"])
    P_t = torch.tensor(P, device=DEVICE)
    for _ in range(c["epochs"]):
        perm = torch.randperm(len(P_t), device=DEVICE)
        for s in range(0, len(P_t), c["batch"]):
            idx = perm[s:s + c["batch"]]
            ctr, ctx_ = P_t[idx, 0], P_t[idx, 1]
            negs = torch.multinomial(noise, len(idx) * c["neg"], replacement=True).view(len(idx), c["neg"])
            vc = emb_in(ctr)
            pos_s = (vc * emb_out(ctx_)).sum(-1)
            neg_s = torch.bmm(emb_out(negs), vc.unsqueeze(-1)).squeeze(-1)
            loss = -(F.logsigmoid(pos_s).mean() + F.logsigmoid(-neg_s).sum(-1).mean())
            opt.zero_grad(); loss.backward(); opt.step()
    return emb_in.weight.detach().cpu().numpy()

def train_node2vec(Gobs, cfg, seed, p=None, q=None):
    c = cfg["n2v"]
    p = c["p"] if p is None else p
    q = c["q"] if q is None else q
    rng = np.random.default_rng(seed); torch.manual_seed(seed)
    walks = node2vec_walks(Gobs, p, q, c["num_walks"], c["walk_len"], rng)
    return train_skipgram(walks, Gobs.number_of_nodes(), c, rng)

def n2v_op(Z, pairs, op):
    a, b = Z[pairs[:, 0]], Z[pairs[:, 1]]
    return {"hadamard": a * b, "l1": np.abs(a - b), "l2": (a - b) ** 2}[op]

OPS = ["hadamard", "l1", "l2"]

def tune_pq(Gobs, st, seed, cfg):
    """Pick (p, q) by average precision on the calibration split (Hadamard logistic regression)."""
    best = (-1.0, (1.0, 1.0))
    for p, q in PQ_GRID:
        Z = train_node2vec(Gobs, cfg, seed, p, q)
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
        clf.fit(n2v_op(Z, st["sup"]["pairs"], "hadamard"), st["sup"]["y"])
        ap = average_precision_score(st["cal"]["y"], clf.predict_proba(n2v_op(Z, st["cal"]["pairs"], "hadamard"))[:, 1])
        if ap > best[0]:
            best = (ap, (p, q))
    print("  tuned (p, q) =", best[1], "cal AP =", round(best[0], 3))
    return best[1]

## 6. GAE and VGAE from scratch

A two-layer GCN encoder maps the observed graph to a mean `μ_i` and log-standard-deviation per node. Setting `variational=False` gives the non-variational GAE, where the variance is zero. Training follows Kipf and Welling: a binary cross-entropy reconstruction loss plus a KL term weighted by `1/n`. The reconstruction uses positives and an equal number of uniformly sampled non-edges, resampled every epoch. This is a deliberate deviation from full-adjacency reconstruction, which is infeasible for PubMed and is discussed at the end. For featureless graphs the first layer is a learned lookup table, which is equivalent to multiplying by the identity matrix without materialising it.

In [ ]:
def normalize_adj(A):
    n = A.shape[0]
    At = A + sp.eye(n)
    d = np.asarray(At.sum(1)).ravel()
    D = sp.diags(d ** -0.5)
    An = (D @ At @ D).tocoo()
    idx = torch.tensor(np.vstack([An.row, An.col]), dtype=torch.long)
    return torch.sparse_coo_tensor(idx, torch.tensor(An.data, dtype=torch.float32), (n, n)).coalesce().to(DEVICE)

class GCNEncoder(nn.Module):
    """Two-layer GCN giving (mu, log sigma) per node."""
    def __init__(self, n, in_dim, hid, dim, featureless):
        super().__init__()
        self.featureless = featureless
        if featureless:
            self.W0 = nn.Parameter(torch.empty(n, hid)); nn.init.xavier_uniform_(self.W0)
        else:
            self.lin0 = nn.Linear(in_dim, hid, bias=False); nn.init.xavier_uniform_(self.lin0.weight)
        self.lin_mu = nn.Linear(hid, dim, bias=False); nn.init.xavier_uniform_(self.lin_mu.weight)
        self.lin_sig = nn.Linear(hid, dim, bias=False); nn.init.xavier_uniform_(self.lin_sig.weight)

    def forward(self, X, Ahat):
        h = self.W0 if self.featureless else self.lin0(X)
        H = F.relu(torch.sparse.mm(Ahat, h))
        return torch.sparse.mm(Ahat, self.lin_mu(H)), torch.sparse.mm(Ahat, self.lin_sig(H))

def sample_nonedges(A, k, rng):
    n = A.shape[0]; out = []
    while len(out) < k:
        u = rng.integers(n, size=2 * k); v = rng.integers(n, size=2 * k)
        ok = (u != v) & (np.asarray(A[u, v]).ravel() == 0)
        out.extend(zip(u[ok], v[ok]))
    return np.array(out[:k], dtype=np.int64)

def train_gae(Gobs, X, cfg, seed, variational=True):
    """Returns (mu, s2) per node. For GAE (variational=False), s2 is all zeros."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    n = Gobs.number_of_nodes(); c = cfg["gae"]
    A = sp.csr_matrix(nx.to_scipy_sparse_array(Gobs, nodelist=list(range(n)), format="csr", dtype=float))
    Ahat = normalize_adj(A)
    featureless = X is None
    Xt = None if featureless else torch.tensor(X, dtype=torch.float32, device=DEVICE)
    enc = GCNEncoder(n, 0 if featureless else X.shape[1], c["hidden"], c["dim"], featureless).to(DEVICE)
    opt = torch.optim.Adam(enc.parameters(), lr=c["lr"])
    E = torch.tensor(np.array(list(Gobs.edges()), dtype=np.int64), device=DEVICE)

    for _ in range(c["epochs"]):
        enc.train()
        mu, logstd = enc(Xt, Ahat)
        sig = torch.exp(logstd)
        z = mu + torch.randn_like(mu) * sig if variational else mu
        Neg = torch.tensor(sample_nonedges(A, len(E), rng), device=DEVICE)
        logits = torch.cat([(z[E[:, 0]] * z[E[:, 1]]).sum(-1), (z[Neg[:, 0]] * z[Neg[:, 1]]).sum(-1)])
        labels = torch.cat([torch.ones(len(E), device=DEVICE), torch.zeros(len(Neg), device=DEVICE)])
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        if variational:
            kl = -(0.5 / n) * torch.mean(torch.sum(1 + 2 * logstd - mu ** 2 - sig ** 2, dim=1))
            loss = loss + kl
        opt.zero_grad(); loss.backward(); opt.step()

    enc.eval()
    with torch.no_grad():
        mu, logstd = enc(Xt, Ahat)
        s2 = torch.exp(2 * logstd) if variational else torch.zeros_like(mu)
    return mu.cpu().numpy(), s2.cpu().numpy()

## 7. Recovering the VGAE posterior: Propositions 1 and 2

VGAE scores a link as `σ(μ_iᵀμ_j)` and ignores `σ_i`. Under the factorised Gaussian posterior, the inner-product score `s_ij = z_iᵀz_j` has:

- **Proposition 1 (exact).** `E[s] = μ_iᵀμ_j` and `Var[s] = ⟨μ_i², σ_j²⟩ + ⟨μ_j², σ_i²⟩ + ⟨σ_i², σ_j²⟩`.
- **Proposition 2 (approximate).** The probit approximation gives `p̃ = σ(m / √(1 + (π/8) v))`. Setting `v = 0` recovers the standard VGAE score.

Both cost `O(d)` per pair. The cell verifies Proposition 1 against a Monte Carlo estimate and checks that Proposition 2 is closer to the sampled `E[σ(s)]` than the plain sigmoid of the mean.

In [ ]:
def inner_product_moments(mu, s2, U, V):
    """Exact mean and variance of z_i^T z_j under the factorised Gaussian posterior (Proposition 1)."""
    m = (mu[U] * mu[V]).sum(1)
    v = ((mu[U] ** 2) * s2[V]).sum(1) + ((mu[V] ** 2) * s2[U]).sum(1) + (s2[U] * s2[V]).sum(1)
    return m, v

def moderated_prob(m, v):
    """Probit-moderated predictive probability (Proposition 2)."""
    return expit(m / np.sqrt(1.0 + (np.pi / 8.0) * v))

def mc_verify(samples=200_000, d=16, seed=0):
    rng = np.random.default_rng(seed)
    mu = rng.normal(size=(2, d)); s2 = rng.uniform(0.01, 0.5, size=(2, d))
    m, v = inner_product_moments(mu, s2, np.array([0]), np.array([1]))
    zi = mu[0] + np.sqrt(s2[0]) * rng.standard_normal((samples, d))
    zj = mu[1] + np.sqrt(s2[1]) * rng.standard_normal((samples, d))
    s = (zi * zj).sum(1)
    return dict(mean_closed=m[0], mean_mc=s.mean(), var_closed=v[0], var_mc=s.var(),
                p_mc=expit(s).mean(), p_moderated=moderated_prob(m, v)[0], p_sigmoid_of_mean=expit(m[0]))

print(mc_verify())

## 8. CALIBRE: four evidence families fused by gradient boosting

For each candidate pair:

- **struct:** node2vec inner product, cosine, Euclidean and L1 distance, endpoint norms, and three stacked logistic-regression scores, one per binary operator (Hadamard, L1, L2). Supervision rows receive 5-fold out-of-fold predictions so the stacker never sees its own training label.
- **gen:** the VGAE posterior-mean score, its sigmoid, and the coordinatewise product `μ_i ⊙ μ_j`.
- **unc:** the seven quantities derived from Propositions 1–2: variance, its square root, signal-to-noise ratio, `p̃`, the moderation gap, and the sum and maximum of endpoint posterior entropies.
- **topo:** the 22 classical indices from section 4.

The four families are concatenated and fed to a gradient-boosted ensemble trained on supervision edges. Every feature is computed from the observed graph, so the classifier is never fitted on contaminated features. Ablations are obtained by dropping families.

In [ ]:
def stacked_n2v(Z, sup, y_sup, queries, seed):
    """One logistic regression per binary operator. Supervision rows get out-of-fold predictions."""
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    oof = np.zeros((len(sup), len(OPS)))
    qout = [np.zeros((len(q), len(OPS))) for q in queries]
    for j, op in enumerate(OPS):
        Xs = n2v_op(Z, sup, op)
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
        oof[:, j] = cross_val_predict(clf, Xs, y_sup, cv=cv, method="predict_proba")[:, 1]
        clf.fit(Xs, y_sup)
        for k, q in enumerate(queries):
            qout[k][:, j] = clf.predict_proba(n2v_op(Z, q, op))[:, 1]
    return oof, qout

def struct_block(Z, pairs, stack):
    a, b = Z[pairs[:, 0]], Z[pairs[:, 1]]
    na, nb = np.linalg.norm(a, axis=1), np.linalg.norm(b, axis=1)
    inner = (a * b).sum(1)
    return np.column_stack([inner, inner / (na * nb + 1e-12), np.linalg.norm(a - b, axis=1),
                            np.abs(a - b).sum(1), np.minimum(na, nb), np.maximum(na, nb), stack])

def gen_block(mu, pairs):
    a, b = mu[pairs[:, 0]], mu[pairs[:, 1]]
    m = (a * b).sum(1)
    return np.column_stack([m, expit(m), a * b])

def node_entropy(s2):
    return 0.5 * np.log(2 * np.pi * np.e * (s2 + 1e-12)).sum(1)

def unc_block(mu, s2, pairs, H):
    U, V = pairs[:, 0], pairs[:, 1]
    m, v = inner_product_moments(mu, s2, U, V)
    p = moderated_prob(m, v)
    return np.column_stack([v, np.sqrt(v), m / np.sqrt(v + 1e-12), p, expit(m) - p,
                            H[U] + H[V], np.maximum(H[U], H[V])])

def make_blocks(pairs, Z, mu, s2, H, stack, topo):
    return {"struct": struct_block(Z, pairs, stack), "gen": gen_block(mu, pairs),
            "unc": unc_block(mu, s2, pairs, H), "topo": topo}

def stack_blocks(Bk, fams):
    return np.hstack([Bk[f] for f in fams])

def fit_gbdt(X, y, seed):
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                          l2_regularization=1.0, random_state=seed).fit(X, y)

# Feature families used by each variant. "full" is CALIBRE itself.
ABLATIONS = {
    "full": ["struct", "gen", "unc", "topo"],
    "w/o unc": ["struct", "gen", "topo"],
    "topo only": ["topo"],
    "gen only": ["gen"],
    "struct only": ["struct"],
    "unc only": ["unc"],
    "struct+topo": ["struct", "topo"],
}

def gae_prob(mu, pairs):
    return expit((mu[pairs[:, 0]] * mu[pairs[:, 1]]).sum(1))

def mp_prob(mu, s2, pairs):
    m, v = inner_product_moments(mu, s2, pairs[:, 0], pairs[:, 1])
    return moderated_prob(m, v)

## 9. Monotone calibration and calibration metrics

Calibration is fitted on the calibration split and applied identically to every probabilistic method. The candidate maps are identity, Platt scaling with a non-negative slope, and isotonic regression. Every candidate is monotone non-decreasing, so ranking metrics cannot be changed by calibration (except for ties). The choice is made by log-loss on a held-out half. Without the non-negativity constraint, a fit to a score that is anti-correlated with the label learns a negative slope and silently inverts the ranking.

Metrics: expected calibration error over 15 equal-mass bins, Brier score, negative log-likelihood, and the Cox regression of the outcome on the log-odds, where a calibrated model gives slope 1 and intercept 0.

In [ ]:
class MonotoneCalibrator:
    """identity | Platt (slope >= 0) | isotonic, chosen by log-loss on a held-out half."""
    def __init__(self, seed=0):
        self.seed = seed

    @staticmethod
    def _fit(kind, p, y):
        if kind == "identity":
            return ("identity", None)
        if kind == "isotonic":
            return ("isotonic", IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip").fit(p, y))
        z = logit(p)
        def nll(th):
            s = th[0] * z + th[1]
            return np.mean(np.logaddexp(0, s) - y * s)
        res = minimize(nll, x0=np.array([1.0, 0.0]), bounds=[(0, None), (None, None)])
        return ("platt", res.x)

    @staticmethod
    def _apply(model, p):
        kind, par = model
        if kind == "identity":
            return p
        if kind == "isotonic":
            return par.predict(p)
        return expit(par[0] * logit(p) + par[1])

    def fit(self, p, y):
        p = np.clip(p, 1e-6, 1 - 1e-6)
        idx = np.random.default_rng(self.seed).permutation(len(p)); h = len(p) // 2
        a_, b_ = idx[:h], idx[h:]
        best = None
        for kind in ["identity", "platt", "isotonic"]:
            m = self._fit(kind, p[a_], y[a_])
            q = np.clip(self._apply(m, p[b_]), 1e-6, 1 - 1e-6)
            loss = log_loss(y[b_], q, labels=[0, 1])
            if best is None or loss < best[0]:
                best = (loss, kind)
        self.kind_ = best[1]
        self.model_ = self._fit(best[1], p, y)
        return self

    def transform(self, p):
        return self._apply(self.model_, np.clip(p, 1e-6, 1 - 1e-6))

def ece_equal_mass(p, y, bins=15):
    o = np.argsort(p); N = len(p)
    return float(sum(len(c) / N * abs(y[c].mean() - p[c].mean())
                     for c in np.array_split(o, bins) if len(c)))

def brier(p, y):
    return float(np.mean((p - y) ** 2))

def nll(p, y):
    return float(log_loss(y, np.clip(p, 1e-9, 1 - 1e-9), labels=[0, 1]))

def cox(p, y):
    z = logit(np.clip(p, 1e-9, 1 - 1e-9)).reshape(-1, 1)
    lr = LogisticRegression(C=1e6, max_iter=5000).fit(z, y)
    return float(lr.coef_[0, 0]), float(lr.intercept_[0])

def reliability_curve(p, y, bins=10):
    o = np.argsort(p)
    return [(float(p[c].mean()), float(y[c].mean())) for c in np.array_split(o, bins) if len(c)]

def rank_metrics(s, y, k):
    neg = np.sort(s[y == 0]); pos_s = s[y == 1]
    ranks = 1 + (len(neg) - np.searchsorted(neg, pos_s, side="right"))
    return dict(mrr=float(np.mean(1.0 / ranks)), hits=float(np.mean(ranks <= k)))

## 10. From scores to decisions: conformal FDR control

For a test pair with score `s`, its conformal p-value compares it with the scores of known non-edges from the calibration split: `p = (1 + #{null ≥ s}) / (N + 1)`. Under the null that the pair is a non-edge and exchangeable with the calibration negatives, `p` is super-uniform. It depends only on rank, so no distributional assumption is needed, and it is unchanged by monotone calibration. Benjamini–Hochberg applied to these p-values controls the false discovery rate of the asserted links. The Mondrian variant computes p-values within joint-degree strata.

In [ ]:
def conformal_pvalues(s_test, s_null, strata_test=None, strata_null=None):
    s_null = np.asarray(s_null, dtype=float); s_test = np.asarray(s_test, dtype=float)
    if strata_test is None:
        ns = np.sort(s_null); N = len(ns)
        greater = N - np.searchsorted(ns, s_test, side="left")
        return (1 + greater) / (N + 1)
    p = np.ones(len(s_test))
    for c in np.unique(strata_test):
        nul = s_null[strata_null == c]
        if len(nul) == 0:
            continue
        mask = strata_test == c
        ns = np.sort(nul)
        greater = len(nul) - np.searchsorted(ns, s_test[mask], side="left")
        p[mask] = (1 + greater) / (len(nul) + 1)
    return p

def benjamini_hochberg(p, alpha):
    m = len(p); order = np.argsort(p)
    thresh = alpha * np.arange(1, m + 1) / m
    ok = np.nonzero(p[order] <= thresh)[0]
    rej = np.zeros(m, dtype=bool)
    if len(ok):
        rej[order[: ok.max() + 1]] = True
    return rej

def fdr_power(s_test, y_test, s_null, alphas, strata_test=None, strata_null=None):
    p = conformal_pvalues(s_test, s_null, strata_test, strata_null)
    out = {}
    for a in alphas:
        rej = benjamini_hochberg(p, a)
        out[f"fdr@{a}"] = float((rej & (y_test == 0)).sum() / max(rej.sum(), 1))
        out[f"power@{a}"] = float((rej & (y_test == 1)).sum() / max((y_test == 1).sum(), 1))
    return out

## 11. Experiment runner

For each dataset, regime, seed, and negative mode, `run_replicate` builds the nested split and the three negative sets. It then trains node2vec and GAE/VGAE on the observed graph only, and records every method's test and calibration scores. Every method sees identical splits, negatives, and seeds, so comparisons are paired. A leakage assertion checks that no supervision, calibration, or test edge appears in the observed graph.

In [ ]:
NEG_IDX = {"uniform": 0, "hard": 1, "hard_degree": 2}

def run_replicate(name, G, X, ts, regime, seed, cfg):
    n = G.number_of_nodes()
    S = split_edges(G, regime, np.random.default_rng([seed, 17]), cfg, ts)
    Gobs = build_graph(n, S["obs"])
    for key in ["sup", "cal", "test"]:
        assert not any(Gobs.has_edge(int(a), int(b)) for a, b in S[key]), "leakage into observed graph"
    ctx = build_context(Gobs, seed)

    sets = {}
    for mode in cfg["neg_modes"]:
        srng = np.random.default_rng([seed, NEG_IDX[mode], 1]); used = set(); sets[mode] = {}
        for key in ["sup", "cal", "test"]:
            pos = S[key]
            neg = sample_negatives(len(pos), mode, G, Gobs, srng, used, pos)
            sets[mode][key] = dict(pairs=np.vstack([pos, neg]),
                                   y=np.r_[np.ones(len(pos)), np.zeros(len(neg))].astype(int))

    first = sets[cfg["neg_modes"][0]]
    pq = (1.0, 1.0) if FAST else tune_pq(Gobs, first, seed, cfg)
    Z = train_node2vec(Gobs, cfg, seed, *pq)
    vg_mu, vg_s2 = train_gae(Gobs, X, cfg, seed, variational=True)
    ga_mu, _ = train_gae(Gobs, X, cfg, seed, variational=False)
    H = node_entropy(vg_s2)

    recs = []
    for mode in cfg["neg_modes"]:
        st = sets[mode]; sup, cal, test = st["sup"], st["cal"], st["test"]
        T = {k: topo_features(st[k]["pairs"], ctx) for k in ["sup", "cal", "test"]}
        oof, (q_cal, q_test) = stacked_n2v(Z, sup["pairs"], sup["y"], [cal["pairs"], test["pairs"]], seed)
        stacks = {"sup": oof, "cal": q_cal, "test": q_test}
        B = {k: make_blocks(st[k]["pairs"], Z, vg_mu, vg_s2, H, stacks[k], T[k]) for k in ["sup", "cal", "test"]}

        P = {}   # method -> (native probability on cal, native probability on test)
        P["node2vec (Hadamard)"] = (q_cal[:, 0], q_test[:, 0])
        P["node2vec (L2)"] = (q_cal[:, 2], q_test[:, 2])
        P["GAE"] = (gae_prob(ga_mu, cal["pairs"]), gae_prob(ga_mu, test["pairs"]))
        P["VGAE"] = (gae_prob(vg_mu, cal["pairs"]), gae_prob(vg_mu, test["pairs"]))
        P["VGAE-MP (Prop. 2)"] = (mp_prob(vg_mu, vg_s2, cal["pairs"]), mp_prob(vg_mu, vg_s2, test["pairs"]))
        for label, fams in ABLATIONS.items():
            model = fit_gbdt(stack_blocks(B["sup"], fams), sup["y"], seed)
            name_ = "CALIBRE" if label == "full" else f"abl: {label}"
            P[name_] = (model.predict_proba(stack_blocks(B["cal"], fams))[:, 1],
                        model.predict_proba(stack_blocks(B["test"], fams))[:, 1])

        methods = {}
        for mname, (pc, pt) in P.items():
            cm = MonotoneCalibrator(seed).fit(pc, cal["y"])
            methods[mname] = dict(rank=pt, native=pt, cal_s=cm.transform(pc), test_s=cm.transform(pt))
        methods["Adamic-Adar"] = dict(rank=T["test"][:, 2], native=None,
                                      cal_s=T["cal"][:, 2], test_s=T["test"][:, 2])
        methods["Resource allocation"] = dict(rank=T["test"][:, 3], native=None,
                                              cal_s=T["cal"][:, 3], test_s=T["test"][:, 3])

        neg_cal = cal["y"] == 0
        strat_test = stratum(test["pairs"], ctx["deg"])
        strat_null = stratum(cal["pairs"][neg_cal], ctx["deg"])
        for mname, d in methods.items():
            r = dict(neg_mode=mode, method=mname)
            r["auc"] = roc_auc_score(test["y"], d["rank"])
            r["ap"] = average_precision_score(test["y"], d["rank"])
            r.update(rank_metrics(d["rank"], test["y"], cfg["hits_k"]))
            if d["native"] is not None:
                r["ece_native"] = ece_equal_mass(d["native"], test["y"])
                r["ece"] = ece_equal_mass(d["test_s"], test["y"])
                r["brier"] = brier(d["test_s"], test["y"])
                r["nll"] = nll(d["test_s"], test["y"])
                r["slope"], r["intercept"] = cox(d["test_s"], test["y"])
                r["curve_native"] = reliability_curve(d["native"], test["y"])
                r["curve_cal"] = reliability_curve(d["test_s"], test["y"])
            r.update(fdr_power(d["test_s"], test["y"], d["cal_s"][neg_cal], cfg["alphas"]))
            r.update({"mondrian_" + k: v for k, v in
                      fdr_power(d["test_s"], test["y"], d["cal_s"][neg_cal], cfg["alphas"],
                                strat_test, strat_null).items()})
            recs.append(r)
    return recs

def run_study(cfg):
    recs = []
    for name in cfg["datasets"]:
        G, X, ts = load_graph(name)
        print(f"{name}: |V|={G.number_of_nodes()} |E|={G.number_of_edges()} "
              f"attributes={'none' if X is None else X.shape[1]}")
        for regime in cfg["regimes"]:
            if regime == "temporal" and ts is None:
                continue
            for seed in cfg["seeds"]:
                t0 = time.time()
                for r in run_replicate(name, G, X, ts, regime, seed, cfg):
                    r.update(dataset=name, regime=regime, seed=seed)
                    recs.append(r)
                print(f"  {regime:10s} seed={seed}  {time.time() - t0:6.1f}s")
    df = pd.DataFrame(recs)
    df.to_csv(cfg["out_dir"] / "replicates.csv", index=False)
    return df

df = run_study(CONFIG)
print(df.groupby(["neg_mode", "method"]).auc.mean().unstack("neg_mode").round(3))

## 12. Paired statistics

Every comparison is paired by dataset, regime, negative mode, and seed. The reported statistics are the one-sided Wilcoxon signed-rank test (CALIBRE against each competitor), Holm–Bonferroni correction across the family of comparisons, a percentile bootstrap confidence interval for the mean paired difference, and the rank-biserial correlation as a distribution-free effect size.

In [ ]:
def holm(pvals):
    p = np.asarray(pvals, dtype=float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i])
        adj[i] = min(run, 1.0)
    return adj

def paired_table(df, metric="auc"):
    return df.pivot_table(index=["dataset", "regime", "neg_mode", "seed"], columns="method", values=metric)

def paired_test(piv, a, b, n_boot=5000, seed=0):
    d = (piv[a] - piv[b]).dropna().values
    d_nz = d[d != 0]
    if len(d_nz) == 0:
        return dict(comparison=f"{a} vs {b}", n=len(d), mean_diff=0.0, ci_low=0.0, ci_high=0.0, p=1.0, rank_biserial=0.0)
    _, p = stats.wilcoxon(d_nz, alternative="greater")
    ranks = stats.rankdata(np.abs(d_nz))
    r = (ranks[d_nz > 0].sum() - ranks[d_nz < 0].sum()) / ranks.sum()
    boots = np.random.default_rng(seed).choice(d, size=(n_boot, len(d)), replace=True).mean(1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return dict(comparison=f"{a} vs {b}", n=len(d), mean_diff=float(d.mean()),
                ci_low=float(lo), ci_high=float(hi), p=float(p), rank_biserial=float(r))

def stat_summary(df, metric="auc", focus="CALIBRE"):
    piv = paired_table(df, metric)
    rows = [paired_test(piv, focus, m) for m in piv.columns
            if m != focus and not str(m).startswith("abl")]
    res = pd.DataFrame(rows)
    res["p_holm"] = holm(res["p"].values)
    return res

stat_auc = stat_summary(df, "auc")
print(stat_auc.round(4).to_string(index=False))

## 13. Tables and figures

Tables are written to `results/` as CSV, with LaTeX output when `jinja2` is available. Figures cover the discrimination comparison under the three negative schemes (Fig. 1), reliability diagrams (Fig. 2), and conformal FDR validity and power (Fig. 3).

In [ ]:
def wide_table(df, metric="auc", neg_mode="hard"):
    sub = df[df.neg_mode == neg_mode]
    return sub.groupby(["method", "dataset"])[metric].mean().unstack("dataset").round(3)

def save_table(t, stem):
    t.to_csv(CONFIG["out_dir"] / f"{stem}.csv")
    try:
        t.to_latex(CONFIG["out_dir"] / f"{stem}.tex", float_format="%.3f")
    except Exception as e:
        print("LaTeX export skipped:", e)

for mode in CONFIG["neg_modes"]:
    save_table(wide_table(df, "auc", mode), f"table_auc_{mode}")
save_table(stat_auc.round(4), "table_paired_tests_auc")

cal_cols = ["ece_native", "ece", "brier", "nll", "slope", "intercept"]
cal_tab = df[df.neg_mode == "uniform"].groupby("method")[cal_cols].mean().round(3)
save_table(cal_tab, "table_calibration_uniform")
print(cal_tab)

def fig_auc_by_negatives(df, methods=("Adamic-Adar", "node2vec (Hadamard)", "VGAE", "CALIBRE")):
    modes = [m for m in CONFIG["neg_modes"] if m in set(df.neg_mode)]
    fig, axes = plt.subplots(1, len(modes), figsize=(4.6 * len(modes), 3.8), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, mode in zip(axes, modes):
        t = (df[(df.neg_mode == mode) & df.method.isin(methods)]
             .groupby(["dataset", "method"]).auc.mean().unstack("method")[list(methods)])
        t.plot.bar(ax=ax, rot=30, width=0.8, legend=(ax is axes[-1]))
        ax.axhline(0.5, ls=":", c="k"); ax.set_ylim(0, 1); ax.set_title(f"{mode} negatives")
        ax.set_ylabel("test AUC"); ax.set_xlabel("")
    plt.tight_layout(); plt.savefig(CONFIG["out_dir"] / "fig1_auc_by_negatives.pdf"); plt.show()

def fig_reliability(df, dataset=None, mode="uniform"):
    dataset = dataset or df.dataset.iloc[0]
    sub = df[(df.dataset == dataset) & (df.neg_mode == mode) & df.method.isin(["VGAE", "CALIBRE"])]
    sub = sub[sub.seed == sub.seed.min()]
    fig, ax = plt.subplots(figsize=(4.6, 4.2)); ax.plot([0, 1], [0, 1], "k--", label="perfect")
    for _, r in sub.iterrows():
        for key, ls, tag in [("curve_native", "--", "native"), ("curve_cal", "-", "recalibrated")]:
            c = np.array(r[key])
            ax.plot(c[:, 0], c[:, 1], ls, marker="o", label=f"{r.method} {tag}")
    ax.set_xlabel("predicted probability"); ax.set_ylabel("observed link frequency")
    ax.set_title(f"Reliability: {dataset}, {mode} negatives"); ax.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(CONFIG["out_dir"] / "fig2_reliability.pdf"); plt.show()

def fig_fdr_power(df, dataset=None, mode="hard"):
    dataset = dataset or df.dataset.iloc[0]
    sub = df[(df.dataset == dataset) & (df.neg_mode == mode)]
    alphas = CONFIG["alphas"]
    methods = ["Adamic-Adar", "node2vec (Hadamard)", "VGAE", "CALIBRE"]
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    for m in methods:
        s = sub[sub.method == m]
        axes[0].plot(alphas, [s[f"fdr@{a}"].mean() for a in alphas], "o-", label=m)
        axes[1].plot(alphas, [s[f"power@{a}"].mean() for a in alphas], "o-", label=m)
    axes[0].plot([0, max(alphas)], [0, max(alphas)], "k--", label="target")
    axes[0].set(xlabel="target FDR", ylabel="realised FDR", title="Validity")
    axes[1].set(xlabel="target FDR", ylabel="power (recall)", title="Power")
    axes[0].legend(fontsize=7)
    plt.tight_layout(); plt.savefig(CONFIG["out_dir"] / "fig3_fdr_power.pdf"); plt.show()

fig_auc_by_negatives(df)
fig_reliability(df)
fig_fdr_power(df)

## 14. Sanity checks

These cover the ten-node values, Proposition 1 and Proposition 2 against simulation, the super-uniformity of conformal p-values under the null, and the behaviour of monotone calibration on an anti-correlated score.

In [ ]:
def sanity_checks(seed=0):
    rng = np.random.default_rng(seed)
    # 1. Ten-node example
    assert abs(aa_manual(TOY, *HARD_FAKE) - 1 / math.log(2)) < 1e-12
    assert aa_manual(TOY, *HIDDEN_REAL) == 0
    print("[ok] ten-node Adamic–Adar values")

    # 2. Proposition 1 vs Monte Carlo
    v = mc_verify()
    assert abs(v["mean_closed"] - v["mean_mc"]) < 0.05
    assert abs(v["var_closed"] - v["var_mc"]) / v["var_closed"] < 0.02
    print("[ok] Proposition 1 (mean, variance):", {k: round(float(x), 4) for k, x in v.items()})

    # 3. Proposition 2 is closer to E[sigmoid(s)] than the plain sigmoid of the mean
    err_mod = abs(v["p_moderated"] - v["p_mc"]); err_naive = abs(v["p_sigmoid_of_mean"] - v["p_mc"])
    print(f"[info] Proposition 2 abs error {err_mod:.5f} vs sigmoid(mean) {err_naive:.5f}")

    # 4. Conformal p-values are super-uniform under the null
    null_scores = rng.random(999)
    pv = np.array([conformal_pvalues(np.array([rng.random()]), null_scores)[0] for _ in range(2000)])
    print(f"[info] P(p <= 0.10 | null) = {np.mean(pv <= 0.10):.3f} (must be <= 0.10 up to noise)")

    # 5. Monotone calibration on an anti-correlated score
    x = rng.normal(size=4000)
    y = (rng.random(4000) < expit(-2 * x)).astype(int)
    p = expit(x)
    cm = MonotoneCalibrator(0).fit(p, y)
    print("[info] anti-correlated score: raw AUC", round(roc_auc_score(y, p), 3),
          "| calibrated AUC", round(roc_auc_score(y, cm.transform(p)), 3), "| chosen map:", cm.kind_)

    # 6. Leakage: supervision/calibration/test edges are not in the observed graph (checked inside run_replicate)
    print("[ok] leakage assertions active in run_replicate")

sanity_checks()